In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Extracción y Remuestreo de Precios del Mercado Secundario (aFRR)

Este script procesa las señales de precio de la reserva de regulación secundaria.
A diferencia del mercado diario, el aFRR tiene una retribución dual:
1. Capacidad (Banda): Retribución marginal horaria por estar disponible.
2. Energía: Retribución marginal cuarto-horaria (15-min) por activación real (AGC).
El código ingesta ambas series, depura los duplicados de la API de ESIOS y
remuestrea los datos de energía a resolución horaria para alinear los vectores
en el optimizador MILP.

Autor: Alberto Zafra Muñoz
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple

# ==========================================
# 1. CONSTANTES Y RUTAS DE ARCHIVOS
# ==========================================
ARCHIVO_BANDA = "Precios_aFRR_Limpios_Horarios.csv"
ARCHIVO_ENE_SUBIR = "Precios_Activación_Potencia_Subir_3_4_2026.csv"
ARCHIVO_ENE_BAJAR = "Precios_Activación_Potencia_Bajar_3_4_2026.csv"

# ==========================================
# 2. PROCESAMIENTO DE DATOS (ETL)
# ==========================================
def procesar_mercado_afrr(
    ruta_banda: str, 
    ruta_subir: str, 
    ruta_bajar: str
) -> Optional[Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Ingesta, limpia y remuestrea los precios de capacidad y energía del aFRR.

    Retorna:
        Tuple[pd.DataFrame, pd.DataFrame]: (df_banda, df_energia_horaria)
        o None si ocurre algún error de lectura.
    """
    print("[*] Iniciando procesamiento de la base de datos aFRR...")
    
    if not (os.path.exists(ruta_banda) and os.path.exists(ruta_subir) and os.path.exists(ruta_bajar)):
        print("[ERROR] Faltan archivos CSV del mercado secundario en el directorio.")
        return None

    try:
        # 1. Carga de datos de Banda (Ya en formato horario)
        df_banda = pd.read_csv(ruta_banda)
        
        # 2. Carga de datos de Energía (Formato cuarto-horario, separador ';')
        df_act_subir = pd.read_csv(ruta_subir, sep=";")
        df_act_bajar = pd.read_csv(ruta_bajar, sep=";")

        # 3. Limpieza y conversión temporal para Energía a SUBIR
        df_act_subir['datetime'] = pd.to_datetime(df_act_subir['datetime'])
        df_act_subir = df_act_subir.drop_duplicates(subset=['datetime'])
        df_act_subir.set_index('datetime', inplace=True)
        # Remuestreo temporal riguroso: de 15-min a medias horarias
        energia_subir_h = df_act_subir['value'].resample('h').mean().reset_index()

        # 4. Limpieza y conversión temporal para Energía a BAJAR
        df_act_bajar['datetime'] = pd.to_datetime(df_act_bajar['datetime'])
        df_act_bajar = df_act_bajar.drop_duplicates(subset=['datetime'])
        df_act_bajar.set_index('datetime', inplace=True)
        # Remuestreo temporal riguroso: de 15-min a medias horarias
        energia_bajar_h = df_act_bajar['value'].resample('h').mean().reset_index()

        # 5. Consolidación de la Energía
        df_energia = pd.DataFrame({
            'Hora': range(1, 25),
            'Energia_SUBIR_EUR_MWh': energia_subir_h['value'],
            'Energia_BAJAR_EUR_MWh': energia_bajar_h['value']
        })

        print("[*] Datos procesados: Remuestreo de 15-min a base horaria completado.")
        return df_banda, df_energia

    except Exception as e:
        print(f"[ERROR INESPERADO] Fallo al procesar el mercado aFRR: {e}")
        return None

# ==========================================
# 3. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_mercados_afrr(df_banda: pd.DataFrame, df_energia: pd.DataFrame):
    """
    Genera la figura con dos paneles: Disponibilidad de Banda y Energía Activada.
    """
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

    horas = range(1, 25)

    # --- PANEL SUPERIOR: MERCADO DE CAPACIDAD (BANDA) ---
    ax1.plot(horas, df_banda['Precio SUBIR (€/MWh)'], 
             marker='o', color='teal', linewidth=2.5, markersize=7,
             label='Retribución Banda a SUBIR ($\lambda_t^{aFRR, s}$)')
    
    ax1.plot(horas, df_banda['Precio BAJAR (€/MWh)'], 
             marker='s', color='orange', linewidth=2.5, markersize=7,
             label='Retribución Banda a BAJAR ($\lambda_t^{aFRR, b}$)')
    
    ax1.set_title('Mercado Secundario (aFRR) - Retribución por Reserva de Capacidad', 
                  fontsize=14, fontweight='bold')
    ax1.set_ylabel('Precio Marginal (€/MWh)', fontsize=12)
    ax1.legend(loc='upper right', fontsize=11, frameon=True, shadow=True)
    ax1.grid(axis='both', linestyle='--', alpha=0.7)

    # --- PANEL INFERIOR: MERCADO DE ENERGÍA ACTIVADA ---
    ax2.plot(horas, df_energia['Energia_SUBIR_EUR_MWh'], 
             marker='o', color='dodgerblue', linewidth=2.5, markersize=7,
             label='Retribución Energía a SUBIR ($\lambda_t^{act, s}$)')
    
    ax2.plot(horas, df_energia['Energia_BAJAR_EUR_MWh'], 
             marker='s', color='crimson', linewidth=2.5, markersize=7,
             label='Retribución Energía a BAJAR ($\lambda_t^{act, b}$)')
    
    ax2.set_title('Mercado Secundario (aFRR) - Retribución por Energía Efectivamente Activada', 
                  fontsize=14, fontweight='bold')
    ax2.set_xlabel('Hora del día ($t$)', fontsize=12)
    ax2.set_ylabel('Precio Marginal (€/MWh)', fontsize=12)
    ax2.legend(loc='upper right', fontsize=11, frameon=True, shadow=True)
    ax2.grid(axis='both', linestyle='--', alpha=0.7)

    # --- FORMATO DE EJES ---
    ax2.set_xticks(horas)
    ax2.set_xlim(0.5, 24.5)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    resultados = procesar_mercado_afrr(ARCHIVO_BANDA, ARCHIVO_ENE_SUBIR, ARCHIVO_ENE_BAJAR)
    
    if resultados is not None:
        datos_banda, datos_energia = resultados
        
        print("\n==================================================================")
        print(" VISTA PREVIA: PRECIOS DE ENERGÍA REMUESTREADOS A 1H (€/MWh)")
        print("==================================================================")
        print(datos_energia.head(5).to_string(index=False))
        print("==================================================================\n")
        
        print("[*] Generando renderizado visual dual...")
        visualizar_mercados_afrr(datos_banda, datos_energia)